In [1]:
import pandas as pd
import re
import os
from ast import literal_eval
from collections import Counter
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import matplotlib.lines as mlines
import matplotlib.ticker as ticker

In [2]:
username = os.path.expanduser('~')
username

'/home/ehdgus7279'

In [3]:
benchmarks = [
    # High-sharing Apps
    'bfs',
    'gemm',
    'lu',
    'spmv',
    'mm3',
    'mm2',
    'matrixmultiplication',
    'syr2k',
    'gups',    
    'mis',
    'matrixtranspose',
    'stencil2d',
    'gaussian',
    'im2col',
    'vgg16',

    # Low-sharing Apps
    'fdtd2d',
    'fastwalshtransform',
    'jacobi1d',
    'relu',
    'color',
    'convolution2d',
    'sssp',
    'fir',
    'fft',
    'bitonicsort',
    'pagerank',
]

In [4]:
num_chiplets = 36
ns = 0.000000001
chip_cols = [str(i) for i in range(36)]

In [5]:
base_dir=f'{username}/mgvm/results/memory-traces'
date = "20260221-223953"
expanded_dir = f"{base_dir}/{date}/expanded"
trace_dir = f"{base_dir}/{date}"
print(f"{base_dir}/{date}")

/home/ehdgus7279/mgvm/results/memory-traces/20260221-223953


In [6]:
clusters_3x3 = {
    'cluster_0_3x3': [0, 1, 2, 6, 7, 8, 12, 13, 14],
    'cluster_1_3x3': [3, 4, 5, 9, 10, 11, 15, 16, 17],
    'cluster_2_3x3': [18, 19, 20, 24, 25, 26, 30, 31, 32],
    'cluster_3_3x3': [21, 22, 23, 27, 28, 29, 33, 34, 35],
}

clusters_2x2 = {
    'cluster_0_2x2': [0, 1, 6, 7],       # 좌측 상단
    'cluster_1_2x2': [2, 3, 8, 9],       # 상단 중앙
    'cluster_2_2x2': [4, 5, 10, 11],     # 우측 상단
    'cluster_3_2x2': [12, 13, 18, 19],   # 좌측 중앙
    'cluster_4_2x2': [14, 15, 20, 21],   # 정중앙
    'cluster_5_2x2': [16, 17, 22, 23],   # 우측 중앙
    'cluster_6_2x2': [24, 25, 30, 31],   # 좌측 하단
    'cluster_7_2x2': [26, 27, 32, 33],   # 하단 중앙
    'cluster_8_2x2': [28, 29, 34, 35]    # 우측 하단
}

clusters_3x3_cols = [k for k in clusters_3x3.keys()]
clusters_2x2_cols = [k for k in clusters_2x2.keys()]

chiplet_to_cluster_2x2 = {}
for cluster_name, chiplet_ids in clusters_2x2.items():
    for chip_id in chiplet_ids:
        chiplet_to_cluster_2x2[chip_id] = cluster_name

chiplet_to_cluster_3x3 = {}
for cluster_name, chiplet_ids in clusters_3x3.items():
    for chip_id in chiplet_ids:
        chiplet_to_cluster_3x3[chip_id] = cluster_name

In [44]:
benchmark = 'gemm'

trace_file = f"{expanded_dir}/{benchmark}_mesh_6x6_memory_trace.csv"

if not os.path.exists(trace_file):
    print(f"No {trace_file}, skip")

df = pd.read_csv(trace_file, skipinitialspace=True)

In [45]:
df

,time,pc,addr,owner,rw,0,1,2,3,4,...,28,29,30,31,32,33,34,35,access_count,shared_chiplets
0,0.01219,300011776,2447167488,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,1
1,0.01219,300011776,2447167488,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,1
2,0.01219,300011776,2447167488,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,1
3,0.01219,300011776,2447167488,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,1
4,0.01219,300011776,2447167488,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39859814,0.01339,300012060,2310312640,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
39859815,0.01339,300012000,2389080640,14,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
39859816,0.01339,300012000,2350409600,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
39859817,0.01339,300012000,2381828608,11,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1


In [ ]:
df.groupby(['pc'])['access_count'].sum()

pc
300011776         288
300011796         288
300011936      185760
300011956       88656
300011964       16563
300011972        2065
300011976        2299
300011984       93169
300012000    19897726
300012004     4232898
300012012       17340
300012016       49986
300012024        7927
300012028        1020
300012032         831
300012036         693
300012040         552
300012044       17808
300012052        1751
300012056       80088
300012060    14018806
300012064     1076832
300012068      116930
300012084           3
Name: access_count, dtype: int64

: 

In [57]:
pc_list = df['pc'].unique()
pc_list = np.sort(pc_list)
pc_list


array([300011776, 300011796, 300011936, 300011956, 300011964, 300011972,
       300011976, 300011984, 300012000, 300012004, 300012012, 300012016,
       300012024, 300012028, 300012032, 300012036, 300012040, 300012044,
       300012052, 300012056, 300012060, 300012064, 300012068, 300012084])

In [ ]:
base_pc = 300011520

In [53]:
pc_list_norm = pc_list - base_pc
pc_list_norm

array([256, 276, 416, 436, 444, 452, 456, 464, 480, 484, 492, 496, 504,
       508, 512, 516, 520, 524, 532, 536, 540, 544, 548, 564])

In [54]:
np.sort([hex(int(x)) for x in pc_list_norm])

array(['0x100', '0x114', '0x1a0', '0x1b4', '0x1bc', '0x1c4', '0x1c8',
       '0x1d0', '0x1e0', '0x1e4', '0x1ec', '0x1f0', '0x1f8', '0x1fc',
       '0x200', '0x204', '0x208', '0x20c', '0x214', '0x218', '0x21c',
       '0x220', '0x224', '0x234'], dtype='<U5')

In [55]:
pc_map = {pc: hex(int(pc - base_pc)) for pc in pc_list}
pc_map

{np.int64(300011776): '0x100',
 np.int64(300011796): '0x114',
 np.int64(300011936): '0x1a0',
 np.int64(300011956): '0x1b4',
 np.int64(300011964): '0x1bc',
 np.int64(300011972): '0x1c4',
 np.int64(300011976): '0x1c8',
 np.int64(300011984): '0x1d0',
 np.int64(300012000): '0x1e0',
 np.int64(300012004): '0x1e4',
 np.int64(300012012): '0x1ec',
 np.int64(300012016): '0x1f0',
 np.int64(300012024): '0x1f8',
 np.int64(300012028): '0x1fc',
 np.int64(300012032): '0x200',
 np.int64(300012036): '0x204',
 np.int64(300012040): '0x208',
 np.int64(300012044): '0x20c',
 np.int64(300012052): '0x214',
 np.int64(300012056): '0x218',
 np.int64(300012060): '0x21c',
 np.int64(300012064): '0x220',
 np.int64(300012068): '0x224',
 np.int64(300012084): '0x234'}